# Capstone pipeline

Full Capstone project requires:
1. Download DeepFashion images to train recognition of cloth type and condition.
2. Download Poshmark and Depop sample data for a few cloth types: price, brand, type, condition
3. Clean (remove missing) and unify (rename attributes to be the same across stores) data
4. Prepare data (extract features, prepare training, validation and testing datasets, as well as create synthetic data to similate conditions)
5. Train (run visual training, run categorical training)

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Finding where the notebook is located
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'README.md').exists() and (PROJECT_ROOT.parent / 'README.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

PYTHON = sys.executable

def run(args, check=True, env=None):
    if isinstance(args, str):
        print(f"$ {args}")
        return subprocess.run(args, shell=True, check=check, env=env)
    cmd = ' '.join(shlex.quote(str(part)) for part in args)
    print(f"$ {cmd}")
    return subprocess.run([str(part) for part in args], check=check, env=env)

print(PROJECT_ROOT)
print(PYTHON)


C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone
C:\ProgramData\anaconda3\python.exe


In [2]:
KAGGLE_DATASETS = [
    ('thusharanair/deepfashion2-original-with-dataframes', 'data/raw/deepfashion/kaggle_thushan'),
    ('vishalbsadanand/deepfashion-1', 'data/raw/deepfashion/kaggle_vishal'),
]

SCRAPE_JOBS = [
    {'query': 'dress', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'pants', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'shorts', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'jeans', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'skirt', 'max_items': 500, 'rate_limit': 2.0},
]

SYNTHETIC_INPUT_DIR = 'data/deepfashion/original'
SYNTHETIC_OUTPUT_DIR = 'data/deepfashion/synthetic_degraded'
SYNTHETIC_VARIATIONS = 5
SYNTHETIC_MAX_IMAGES = 5000

CLOTHING_TYPE_DIR = 'data/processed/clothing_type'
CONDITION_DIR = 'data/processed/condition_assessment'
MULTITASK_DIR = 'data/processed/multitask'
FEATURES_DIR = 'data/features'
EMBEDDINGS_DIR = 'data/embeddings'
PRICE_DATASET_DIR = 'data/price_classification'
VECTOR_INDEX_DIR = 'data/vector_index'

UNIFIED_DATA_CSV = 'data/scraped/unified/unified_data.csv'

CLOTHING_TYPE_MODEL_DIR = 'models/clothing_type'
CONDITION_MODEL_DIR = 'models/condition_assessment'
MULTITASK_MODEL_DIR = 'models/multitask'
PRICE_MODEL_DIR = 'models/price_model'

VISION_MODEL_FOR_FEATURES = f'{MULTITASK_MODEL_DIR}/checkpoints/best_loss.pth'

TRAIN_DEVICE = 'cuda'

In [3]:
run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt'])
run([PYTHON, '-m', 'pip', 'install', 'playwright'])
run([PYTHON, '-m', 'playwright', 'install', 'chromium'])

$ 'C:\ProgramData\anaconda3\python.exe' -m pip install -r requirements.txt


CalledProcessError: Command '['C:\\ProgramData\\anaconda3\\python.exe', '-m', 'pip', 'install', '-r', 'requirements.txt']' returned non-zero exit status 1.

## Warning! Next operation will download ~56GBs of images

## Download

In [ ]:
run([
    PYTHON,
    'scripts/download_deepfashion.py',
    '--subset', 'all',
    '--download',
    '--extract',
    '--verify',
])
for dataset, output_dir in KAGGLE_DATASETS:
    run([
        PYTHON,
        'scripts/download_kaggle_datasets.py',
        '--dataset', dataset,
        '--output', output_dir,
    ])

run([
    PYTHON,
    'scripts/process_deepfashion_to_csv.py',
    '--input', 'data/raw/deepfashion',
    '--output', 'data/processed',
])

run([
    PYTHON,
    'scripts/generate_statistics.py',
    '--processed-dir', 'data/processed',
    '--output', 'data/processed/deepfashion_statistics.csv',
])


## Scrape

In [ ]:
for job in SCRAPE_JOBS:
    run([
        PYTHON,
        'scripts/scrape_marketplaces.py',
        '--query', job['query'],
        '--all-platforms',
        '--max-items', str(job['max_items']),
        '--rate-limit', str(job['rate_limit']),
        '--output-dir', 'data/raw/scraped',
    ])


## Clean And Organize

In [ ]:
run([
    PYTHON,
    'scripts/clean_scraped_data.py',
    '--input-dir', 'data/raw/scraped',
    '--output-dir', 'data/scraped',
    '--remove-no-images',
])

run([PYTHON, 'scripts/organize_data.py', '--create-dirs'])

organize_cmd = [PYTHON, 'scripts/organize_data.py', '--organize-deepfashion', '--copy']
run(organize_cmd)

run([PYTHON, 'scripts/organize_data.py', '--verify'])
run([PYTHON, 'scripts/organize_data.py', '--unify', '--format', 'both'])
run([
    PYTHON,
    'scripts/organize_data.py',
    '--summary',
    '--output', 'data/data_organization_report.json',
])


## Prepare

In [ ]:
synthetic_cmd = [
    PYTHON,
    'scripts/generate_synthetic_data.py',
    '--input-dir', SYNTHETIC_INPUT_DIR,
    '--output-dir', SYNTHETIC_OUTPUT_DIR,
    '--num-variations', str(SYNTHETIC_VARIATIONS),
    '--save-metadata',
    '--max-images', str(SYNTHETIC_MAX_IMAGES),
]
run(synthetic_cmd)

run([
    PYTHON,
    'scripts/prepare_splits.py',
    '--all',
    '--clean',
    '--stratify', 'condition',
    '--report', 'data/processed/split_report.json',
])

run([
    PYTHON,
    'scripts/prepare_clothing_type_dataset.py',
    '--deepfashion-root', 'data/deepfashion',
    '--output-dir', CLOTHING_TYPE_DIR,
])

run([
    PYTHON,
    'scripts/prepare_condition_dataset.py',
    '--deepfashion-dir', 'data/deepfashion/original',
    '--synthetic-dir', SYNTHETIC_OUTPUT_DIR,
    '--output-dir', CONDITION_DIR,
])

run([
    PYTHON,
    'scripts/prepare_multitask_dataset.py',
    '--deepfashion2-csv', 'data/processed/deepfashion2_processed.csv',
    '--synthetic-root', SYNTHETIC_OUTPUT_DIR,
    '--output-dir', MULTITASK_DIR,
])


## Train Vision Models

In [ ]:
clothing_type_cmd = [
    PYTHON,
    'scripts/train_clothing_type.py',
    '--data-dir', CLOTHING_TYPE_DIR,
    '--output-dir', CLOTHING_TYPE_MODEL_DIR,
    '--device', TRAIN_DEVICE,
]
run(clothing_type_cmd)

condition_cmd = [
    PYTHON,
    'scripts/train_condition_assessment.py',
    '--train-csv', f'{CONDITION_DIR}/train.csv',
    '--val-csv', f'{CONDITION_DIR}/val.csv',
    '--data-dir', str(PROJECT_ROOT),
    '--output-dir', CONDITION_MODEL_DIR,
]
run(condition_cmd)

multitask_cmd = [
    PYTHON,
    'scripts/train_multitask_model.py',
    '--train-csv', f'{MULTITASK_DIR}/train.csv',
    '--val-csv', f'{MULTITASK_DIR}/val.csv',
    '--output-dir', MULTITASK_MODEL_DIR,
]
run(multitask_cmd)


## Train Price Model

In [ ]:
extract_cmd = [
    PYTHON,
    'scripts/extract_features.py',
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', FEATURES_DIR,
    '--batch-size', '32',
    '--num-workers', '0',
    '--device', TRAIN_DEVICE,
]
if Path(VISION_MODEL_FOR_FEATURES).exists():
    extract_cmd += ['--vision-model', VISION_MODEL_FOR_FEATURES]
else:
    extract_cmd.append('--no-vision')
run(extract_cmd)

run([
    PYTHON,
    'scripts/prepare_price_dataset.py',
    '--features-dir', FEATURES_DIR,
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', PRICE_DATASET_DIR,
    '--strategy', 'quantile',
    '--n-bins', '5',
])

price_cmd = [
    PYTHON,
    'scripts/train_price_model.py',
    '--features-dir', PRICE_DATASET_DIR,
    '--model', 'xgboost',
    '--output-dir', PRICE_MODEL_DIR,
    '--verbose',
]
run(price_cmd)


## Generate Embeddings

In [ ]:
embedding_mode = 'multimodal' if Path(VISION_MODEL_FOR_FEATURES).exists() else 'text'
embedding_cmd = [
    PYTHON,
    'scripts/generate_embeddings.py',
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', EMBEDDINGS_DIR,
    '--mode', embedding_mode,
    '--batch-size', '32',
    '--device', TRAIN_DEVICE,
]
if Path(VISION_MODEL_FOR_FEATURES).exists():
    embedding_cmd += ['--vision-model', VISION_MODEL_FOR_FEATURES]
run(embedding_cmd)

## Build Vector Index

In [ ]:
index_cmd = [
    PYTHON,
    'scripts/build_vector_index.py',
    '--embeddings-dir', EMBEDDINGS_DIR,
    '--output-dir', VECTOR_INDEX_DIR,
    '--output-name', 'clothing_index',
]
run(index_cmd)
